## 1. Introdução ao dataset

O presente estudo realiza uma análise exploratória do conjunto de dados "Air Quality UCI". Este dataset é de grande relevância para a área de ciências ambientais e aprendizado de máquina, pois contém registros detalhados da qualidade do ar de uma cidade metropolitana italiana, coletados entre março de 2004 e fevereiro de 2005.

Os dados foram capturados por um dispositivo multisensor que media, em intervalos horários, a concentração de diversos poluentes atmosféricos, como monóxido de carbono (CO), benzeno (C6H6) e óxidos de nitrogênio (NOx), além de registrar condições meteorológicas como temperatura e umidade.

A análise deste conjunto de dados visa identificar padrões, distribuições e correlações entre as variáveis, fornecendo insights valiosos sobre a dinâmica da poluição do ar em um ambiente urbano. O entendimento dessas características é o primeiro passo para a construção de modelos preditivos e para a avaliação do impacto ambiental na saúde pública.

## 2. Descrição dataset

#### 2.1 Importando bibliotecas

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import sys
import warnings

#### 2.2 Configurações

In [ ]:
# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
%matplotlib inline

warnings.filterwarnings('ignore')
# Add parent directory to sys.path for module imports
sys.path.append('..')

#### 2.2 Carregando dataset

In [ ]:
# Carrega o dataset 
data_path = "../database/air_quality_uci.csv"
air_quality = pd.read_csv(data_path,
                          sep=";",
                          decimal=",", 
                          na_values=-200,
                          date_format="%d/%m/%Y %H.%M.%S",
                          parse_dates={"datetime": ["Date", "Time"]}, 
                          index_col="datetime")  

# Converting 'datetime' column to datetime format with error coercion
air_quality.index = pd.to_datetime(air_quality.index, format="%d/%m/%Y %H.%M.%S", errors="coerce")

# Remoção de duas colunas vazias que estão no final do dataset
air_quality = air_quality.iloc[:, :-2]

# Exibir 5 linhas aleatórias para verificar se foi carregado corretamente
print(f"Dataset shape: {air_quality.shape}")
print("Cabeçalho do DataFrame:")
air_quality.sample(5)

#### 2.3 Obter número de instâncias e variáveis

In [ ]:
print("Dimensões:", air_quality.shape)  # linhas e colunas

In [ ]:
print("Valores nulos por coluna:\n")
air_quality.isna().sum()

#### 2.4 Listagem de variáveis

In [ ]:
# Lista variáveis, tipos de dados e contagem de valores não nulos
print("Dataset Information:")
print("=" * 80)
air_quality.info()


#### 2.5 Contagem de valores nulos por variáveis

In [ ]:
print("\nMissing Values:")
air_quality.isnull().sum()

In [ ]:
print(f"\nTotal missing values: {air_quality.isnull().sum().sum()}")

### 2.6 Apagar linhas duplicadas

In [ ]:
# drop duplicate rows
air_quality.drop_duplicates(inplace=True)

# 3. Estátisticas univariadas incondicionais

#### 3.1 Calculo da media
<span style="font-size: 15px;">A média, ou valor médio, é a medida de tendência central mais comum. No contexto do nosso dataset, ela nos dará um valor "típico" para cada sensor e medição ao longo de todo o período de coleta. </span>

In [ ]:
# Cálculo da média para cada variável numérica
print("Média para cada variável:")
air_quality.mean(numeric_only=True)

#### 3.2 Calculo do desvio padrao
<span style="font-size: 15px;">O desvio padrão nos diz o quão espalhados ou variáveis os dados são em torno dessa média. Um desvio padrão baixo significa que a maioria dos valores está muito próxima da média. Os dados são consistentes e previsíveis. Um desvio padrão alto significa que os valores estão muito espalhados. Há uma grande variação nas medições, com muitos valores longe da média. 
Para o nosso dataset, o desvio padrão vai nos ajudar a entender a volatilidade de cada poluente e condição climática.</span>

In [ ]:
#Cálculo do desvio padrão para cada variável numérica
print("\nDesvio padrão para cada variável:")
air_quality.std(numeric_only=True)

#### 3.3 Describe

In [ ]:
air_quality.describe()

#### 3.4 Calculo de skewness (Assimetria)
Enquanto a média e o desvio padrão nos dão informações sobre o centro e a dispersão, a skewness nos diz sobre a forma da distribuição.
- Skewness ≈ 0: A distribuição é quase simétrica. Os dados se espalham de maneira uniforme em ambos os lados da média. A curva parece um sino (distribuição normal).
- Skewness > 0: A distribuição tem uma "cauda" mais longa à direita. Isso significa que, embora a maioria dos valores esteja concentrada à esquerda, existem alguns valores extremamente altos que "puxam" a média para a direita.
- Skewness < 0: A distribuição tem uma "cauda" mais longa à esquerda. A maioria dos valores está concentrada à direita, mas alguns valores extremamente baixos "puxam" a média para a esquerda.
No nosso caso, a Skewness vai nos dizer, por exemplo, se os picos de poluição são mais comuns do que períodos de ar excepcionalmente limpo.

In [ ]:
# Calcula a skewness para todas as colunas numéricas
print("Assimetria (Skewness) para cada variável:")
air_quality.skew(numeric_only=True)

### 3.4 Plotagem dos gráficos

In [ ]:
# Somente colunas numéricas
numeric_cols = air_quality.copy().select_dtypes(include=np.number).columns

# Define o layout da grade e o tamanho total da figura
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(20, 16))

# Loop para criar histogramas para cada coluna numérica
for col, ax in zip(numeric_cols, axes.flatten()):
    sns.histplot(data=air_quality, x=col, kde=True, bins=30, ax=ax)
    ax.set_title(f'Distribuição de {col}', fontsize=12)


# Esconde as colunas vazias
for i in range(len(numeric_cols), 16):
    axes.flatten()[i].axis('off')

# Espaçamento
plt.tight_layout()

In [ ]:
# Somente as colunas numéricas
numeric_cols = air_quality.select_dtypes(include=np.number).columns

# Define o layout da grade e o tamanho total da figura
fig, axes = plt.subplots(nrows=4, ncols=4, figsize=(20, 16))

# Loop para criar box plots para cada coluna numérica
for col, ax in zip(numeric_cols, axes.flatten()):
    sns.boxplot(x=air_quality[col], ax=ax)
    ax.set_title(f'Box Plot de {col}', fontsize=12)

# Esconde as colunas vazias
for i in range(len(numeric_cols), 16):
    axes.flatten()[i].axis('off')

# Espaçamento
plt.tight_layout()

In [ ]:
# Selecionar apenas colunas numéricas
cols = air_quality.select_dtypes("number").columns

# Criar figura
fig, axes = plt.subplots(nrows=len(cols), ncols=2, figsize=(14, len(cols) * 1.8),
                         gridspec_kw={"width_ratios": [4, 1]})
fig.suptitle("Air Quality Summary Plot (Simplified)", fontsize=16)

for i, col in enumerate(cols):
    # --- Série temporal ---
    ax_ts = axes[i, 0]
    air_quality[col].plot(ax=ax_ts, lw=0.8, color="orange", label=col)
    ax_ts.set_ylabel(col, fontsize=8)
    ax_ts.set_xlabel("")
    ax_ts.legend(loc="upper left", fontsize=7)
    ax_ts.grid(True, alpha=0.3)
    
    # Destacar valores faltantes
    na_mask = air_quality[col].isna()
    ax_ts.scatter(air_quality.index[na_mask], [air_quality[col].mean()]*na_mask.sum(),
                  color="red", s=5, label="missing")
    
    # --- Histograma ---
    ax_hist = axes[i, 1]
    air_quality[col].hist(ax=ax_hist, bins=30, color="green", alpha=0.7)
    ax_hist.set_ylabel("")
    ax_hist.set_xlabel("value", fontsize=8)
    ax_hist.grid(False)
    ax_hist.tick_params(axis="y", labelleft=False)

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()


In [ ]:
air_quality_targets = air_quality[
    [
        "CO(GT)",      # Monóxido de carbono (mg/m³)
        "NMHC(GT)",    # Hidrocarbonetos não metânicos (µg/m³)
        "C6H6(GT)",    # Benzeno (µg/m³)
        "NOx(GT)",     # Óxidos de nitrogênio totais (ppb)
        "NO2(GT)"      # Dióxido de nitrogênio (µg/m³)
    ]
]

air_quality_predictors = air_quality[
    [
        "PT08.S1(CO)",     # Sensor 1 — SnO₂ (CO)
        "PT08.S2(NMHC)",   # Sensor 2 — TiO₂ (NMHC)
        "PT08.S3(NOx)",    # Sensor 3 — WO₃ (NOx)
        "PT08.S4(NO2)",    # Sensor 4 — WO₃ (NO₂)
        "PT08.S5(O3)",     # Sensor 5 — In₂O₃ (O₃)
        "T",               # Temperatura (°C)
        "RH",              # Umidade relativa (%)
        "AH"               # Umidade absoluta (g/m³)
    ]
]

In [ ]:
air_quality

# 4. Análise Monovariada condicionada por classes